# NEQR quantum image watermarking - walkthrough

This notebook runs the worked example of the reference design through every stage of the framework, then watermarks a full image and evaluates it.

In [ ]:
import numpy as np
from qwm.neqr import neqr_block_circuit, encode_decode_roundtrip
from qwm.watermark import block_trace, embed_image, extract_image, generate_watermark, bit_error_rate
from qwm.circuits import WatermarkGenerationCircuit, WatermarkEmbeddingCircuit, WatermarkExtractionCircuit
from qwm.metrics import quality_report, image_metrics
from qwm.attacks import apply_attack, ATTACK_NAMES
from qwm.data import load_image

block = np.array([[50, 60], [48, 32]])
block

## 1. NEQR representation of a 2x2 block

In [ ]:
qc = neqr_block_circuit(block)
print(qc.num_qubits, 'qubits')
qc.draw('mpl', fold=-1)

In [ ]:
encode_decode_roundtrip(block)

## 2. Watermark generation (PPB + AA)

In [ ]:
block_trace(block)

In [ ]:
gen = WatermarkGenerationCircuit(block)
gen.run()

## 3. Embedding and extraction circuits

In [ ]:
wb = gen.run()['watermark']
emb = WatermarkEmbeddingCircuit(block, wb)
wblock = emb.run()['watermarked_block']
ext = WatermarkExtractionCircuit(wblock, wb)
print('watermarked block:', wblock.tolist())
print('extraction:', ext.run())

In [ ]:
emb.circuit.draw('mpl', fold=-1, scale=0.5)

## 4. Full image

In [ ]:
cover = load_image('camera', 256)
for mode in ('paper', 'lsb'):
    watermarked, bits = embed_image(cover, mode=mode)
    raw, _ = extract_image(watermarked, bits)
    print(mode, quality_report(cover, watermarked), 'BER', bit_error_rate(raw, bits))

In [ ]:
import matplotlib.pyplot as plt
watermarked, bits = embed_image(cover, mode='lsb')
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (title, img) in zip(axes, [('cover', cover), ('watermarked', watermarked), ('|cover - watermarked| x 60', np.abs(cover.astype(int) - watermarked.astype(int)) * 60)]):
    ax.imshow(img, cmap='gray', vmin=0, vmax=255); ax.set_title(title); ax.axis('off')
plt.show()

## 5. Attacks and blind extraction

In [ ]:
for name in ATTACK_NAMES:
    attacked = apply_attack(watermarked, name, seed=0)
    raw, _ = extract_image(attacked, bits)
    print(f'{name:24s} PSNR {quality_report(cover, attacked)["psnr"]:6.2f} dB   BER {bit_error_rate(raw, bits):.3f}')

## 6. Quantum-state metrics under a depolarising channel

In [ ]:
image_metrics(cover, watermarked, 'depolarizing', 0.05, max_blocks=4)